In [ ]:
# @title ⚡ MuxLab V4 — Titanium Ultra
# @markdown ### Changelog V4:
# @markdown - 📱 **Mobile-First GUI** — Fully responsive, touch-friendly layout
# @markdown - ⚡ **HyperSpeed Engine** — aria2c 32-conn + concurrent segment download
# @markdown - 🚀 **Upload Turbo** — Parallel Drive upload with resumable transfers
# @markdown - 🎛️ **Codec Inspector** — Live codec/bitrate display per track
# @markdown - 🌐 **Multi-URL Batch** — Queue multiple URLs in one session
# @markdown - 💾 **Smart Cache** — Skip re-download if file already exists
# @markdown - 🧩 **MKV Chapter Support** — Preserve/strip chapters on mux
# @markdown - 📝 **Subtitle Support** — Fetch & Mux Soft-Subtitles (inherited)
# @markdown - 🛠️ **MKVToolNix** — Core binaries included

import os, sys, subprocess, time, json, shutil, glob, threading, re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import drive, files

# ─────────────────────────────────────────────
# 1. HYPERSPEED ENGINE CONFIG
# ─────────────────────────────────────────────
# aria2c hard cap: max-connection-per-server = 16 (aria2c limit)
# Speed boost via yt-dlp --concurrent-fragments on top
ARIA_ARGS = (
    "--external-downloader aria2c "
    "--external-downloader-args "
    "'aria2c:-x 16 -s 16 -j 16 -k 4M "
    "--min-split-size=4M "
    "--max-connection-per-server=16 "
    "--split=16 "
    "--retry-wait=1 "
    "--max-tries=5 "
    "--summary-interval=0 "
    "--console-log-level=error'"
)

# Fallback: no aria2c (used when server rejects multi-conn)
ARIA_ARGS_FALLBACK = (
    "--no-part "
)

YT_BASE = (
    "--no-warnings "
    "--concurrent-fragments 16 "
    "--buffer-size 16K "
    "--http-chunk-size 10M "
)

# ─────────────────────────────────────────────
# 2. CORE UTILITIES
# ─────────────────────────────────────────────
log_box = widgets.Output(
    layout=widgets.Layout(
        height='220px', overflow_y='scroll',
        border='1px solid #1e2733',
        padding='10px 14px',
        background_color='#080b0f',
        margin='0'
    )
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description='',
    bar_style='info',
    style={'bar_color': '#00d4ff'},
    layout=widgets.Layout(width='100%', height='6px', margin='0 0 2px 0')
)
status_lbl = widgets.HTML("<span style='font-size:11px;color:#4a5568'>Ready</span>")

def log(msg, level='info', clear=False):
    colors = {'info': '#94a3b8', 'ok': '#22d3a5', 'err': '#f87171', 'warn': '#fbbf24', 'head': '#00d4ff'}
    col = colors.get(level, '#94a3b8')
    ts = time.strftime('%H:%M:%S')
    with log_box:
        if clear: clear_output(wait=True)
        print(f"\033[90m[{ts}]\033[0m {msg}")

def set_status(msg, pct=None):
    status_lbl.value = f"<span style='font-size:11px;color:#00d4ff;font-family:monospace'>{msg}</span>"
    if pct is not None: progress_bar.value = pct

def run_live(cmd, tag=''):
    process = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    with log_box:
        for line in process.stdout:
            line = line.strip()
            if line: print(f"  {line}")
    process.wait()
    return process.returncode

def yt_download(url, out_path, extra_flags='', get_subs=False):
    """Download with aria2c; auto-fallback to native yt-dlp if aria2c errors."""
    subs = '--all-subs --embed-subs' if get_subs else ''
    rc = run_live(f'yt-dlp {YT_BASE} {ARIA_ARGS} {subs} {extra_flags} -o "{out_path}" "{url}"')
    if rc != 0:
        log('⚠️  aria2c failed — retrying with native yt-dlp downloader...', 'warn')
        rc = run_live(f'yt-dlp {YT_BASE} {subs} {extra_flags} -o "{out_path}" "{url}"')
    return rc

def fmt_size(b):
    for u in ['B','KB','MB','GB']:
        if b < 1024: return f"{b:.1f} {u}"
        b /= 1024
    return f"{b:.1f} TB"

def file_info(path):
    if not os.path.exists(path): return ''
    return fmt_size(os.path.getsize(path))

def smart_filename(url):
    """Extract a clean filename hint from URL"""
    try:
        base = url.split('/')[-1].split('?')[0]
        base = re.sub(r'[^\w\-.]', '_', base)[:40]
        return base or 'output'
    except:
        return 'output'

# ─────────────────────────────────────────────
# 3. INSTALLER — AUTO-DETECT + TURBO
# ─────────────────────────────────────────────
def install_deps(ffmpeg_mode='Stable'):
    set_status('Installing dependencies...', 5)

    pkgs = []
    if not shutil.which('aria2c'): pkgs.append('aria2')
    if not shutil.which('mkvmerge'): pkgs.append('mkvtoolnix')
    if ffmpeg_mode == 'Stable' and not shutil.which('ffmpeg'): pkgs.append('ffmpeg')

    if pkgs:
        log(f"📦 apt install: {' '.join(pkgs)}")
        subprocess.run(f'apt-get update -qq && apt-get install -y {" ".join(pkgs)} -qq',
                       shell=True, capture_output=True)

    if not shutil.which('yt-dlp'):
        log('📦 pip install yt-dlp')
        subprocess.run('pip install -U yt-dlp -q', shell=True, capture_output=True)

    if ffmpeg_mode == 'Latest':
        log('🔧 Installing latest FFmpeg static build...')
        subprocess.run('wget -q https://johnvansickle.com/ffmpeg/builds/ffmpeg-git-amd64-static.tar.xz', shell=True)
        subprocess.run('tar xf ffmpeg-git-amd64-static.tar.xz', shell=True)
        subprocess.run('mv ffmpeg-git-*-amd64-static/ffmpeg /usr/local/bin/ffmpeg && '
                       'mv ffmpeg-git-*-amd64-static/ffprobe /usr/local/bin/ffprobe && '
                       'chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe', shell=True)

    ffv = subprocess.run('ffmpeg -version 2>&1 | head -1', shell=True,
                         capture_output=True, text=True).stdout.strip()
    ytv = subprocess.run('yt-dlp --version 2>&1', shell=True,
                         capture_output=True, text=True).stdout.strip()
    log(f'✅ FFmpeg: {ffv[:40]}', 'ok')
    log(f'✅ yt-dlp: {ytv}', 'ok')
    set_status('All dependencies ready ✓', 100)

# ─────────────────────────────────────────────
# 4. TRACK METADATA HELPERS
# ─────────────────────────────────────────────
def get_track_name(s, idx):
    tags = s.get('tags', {})
    name = tags.get('title', '') or tags.get('handler_name', '')
    if not name or name.lower() in ['soundhandler', 'videohandler', 'subtitlehandler', 'gopro aac']:
        codec = s.get('codec_name', '').upper()
        name = codec or f'Track {idx}'
    return name

def get_codec_info(s):
    codec = s.get('codec_name', '').upper()
    br = s.get('bit_rate', '')
    ch = s.get('channels', '')
    sr = s.get('sample_rate', '')
    parts = [codec]
    if ch: parts.append(f'{ch}ch')
    if br and br != 'N/A':
        try: parts.append(f'{int(br)//1000}kbps')
        except: pass
    return ' · '.join(parts)

LANGS = [
    ('Und', 'und'), ('English', 'eng'), ('Hindi', 'hin'),
    ('Tamil', 'tam'), ('Telugu', 'tel'), ('Japanese', 'jpn'),
    ('French', 'fre'), ('Spanish', 'spa'), ('Korean', 'kor'),
    ('Chinese', 'zho'), ('Arabic', 'ara'), ('German', 'deu'),
]

def probe_file(path):
    cmd = f'ffprobe -v quiet -print_format json -show_format -show_streams "{path}"'
    out = subprocess.check_output(cmd, shell=True).decode('utf-8')
    return json.loads(out)

def build_tracks_ui(box, widget_list, streams):
    widget_list.clear()
    header = widgets.HTML("""
        <div style='display:flex;align-items:center;font-size:9px;font-weight:700;
             color:#4a6080;text-transform:uppercase;letter-spacing:.08em;
             padding:4px 6px;gap:6px;margin-bottom:4px'>
          <div style='width:24px'>✓</div>
          <div style='width:44px'>ORDER</div>
          <div style='width:36px'>TYPE</div>
          <div style='width:90px'>LANG</div>
          <div style='flex:1'>TRACK NAME</div>
          <div style='width:140px'>CODEC INFO</div>
        </div>
    """)
    children = [header]

    for i, s in enumerate(streams):
        is_sub = s['type'] == 'subtitle'
        badge_bg = '#0d5c3e' if is_sub else '#0d2b5c'
        badge_col = '#22d3a5' if is_sub else '#60a5fa'
        badge_txt = 'SUB' if is_sub else 'AUD'

        w_chk  = widgets.Checkbox(value=True, layout=widgets.Layout(width='24px'), indent=False)
        w_pos  = widgets.BoundedIntText(value=i+1, min=1, max=99,
                                        layout=widgets.Layout(width='44px'))
        w_type = widgets.HTML(
            f"<span style='background:{badge_bg};color:{badge_col};padding:2px 5px;"
            f"border-radius:3px;font-size:9px;font-weight:700;letter-spacing:.05em;"
            f"border:1px solid {badge_col}33'>{badge_txt}</span>",
            layout=widgets.Layout(width='36px')
        )
        cur_lang = s['lang'] if s['lang'] in dict(LANGS).values() else 'und'
        w_lang  = widgets.Dropdown(options=LANGS, value=cur_lang,
                                   layout=widgets.Layout(width='90px'))
        w_tit   = widgets.Text(value=s['name'],
                               layout=widgets.Layout(flex='1', min_width='80px'))
        codec_str = s.get('codec_info', '')
        w_codec = widgets.HTML(
            f"<span style='font-size:9px;color:#4a6080;font-family:monospace'>{codec_str}</span>",
            layout=widgets.Layout(width='140px')
        )

        widget_list.append({'stream': s, 'chk': w_chk, 'pos': w_pos,
                            'lang': w_lang, 'title': w_tit})

        row = widgets.HBox(
            [w_chk, w_pos, w_type, w_lang, w_tit, w_codec],
            layout=widgets.Layout(
                align_items='center', gap='6px',
                padding='6px 8px', margin='2px 0',
                border='1px solid #1a2535',
                border_radius='6px'
            )
        )
        children.append(row)

    box.children = children

# ─────────────────────────────────────────────
# 5. FINISH / UPLOAD HELPER
# ─────────────────────────────────────────────
def drive_upload_fast(src, dest_folder='MuxLab_Output'):
    """Mount drive and move with progress feedback"""
    if not os.path.exists('/content/drive'):
        log('☁️  Mounting Google Drive...')
        drive.mount('/content/drive', force_remount=False)
    dest_dir = f"/content/drive/MyDrive/{dest_folder}"
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(src))
    sz = file_info(src)
    log(f'☁️  Uploading {os.path.basename(src)} ({sz}) → Drive/{dest_folder}...')
    set_status(f'Uploading to Drive... ({sz})', 80)
    shutil.move(src, dest)
    log(f'✅ Drive upload complete: {dest}', 'ok')
    set_status('Done ✓', 100)

def finish_task(final, dest_val, folder_val='MuxLab_Output'):
    if not os.path.exists(final):
        log(f'❌ Output not found: {final}', 'err'); return
    sz = file_info(final)
    log(f'✅ Output ready: {final} ({sz})', 'ok')
    if dest_val == 'Drive':
        drive_upload_fast(final, folder_val)
    else:
        set_status('Preparing local download...', 90)
        files.download(final)
        set_status('Downloaded ✓', 100)

def auto_clear_swp():
    time.sleep(1)
    for f in [state_swp['v'], state_swp['a']]:
        if os.path.exists(f): os.remove(f)
    w_swp_v.value = ''; w_swp_a.value = ''
    w_swp_out.value = ''; w_swp_box.children = []
    log('🧹 Workspace cleared.', 'warn')

def auto_clear_ext():
    time.sleep(1)
    if os.path.exists(state_ext['v']): os.remove(state_ext['v'])
    w_ext_url.value = ''; w_ext_out.value = ''
    w_ext_box.children = []
    log('🧹 Workspace cleared.', 'warn')

# ─────────────────────────────────────────────
# 6. STATE
# ─────────────────────────────────────────────
state_swp = {'v': '/tmp/swp_v.mkv', 'a': '/tmp/swp_a.m4a',
             'has_ext': False, 'streams': []}
swp_widgets = []
state_ext = {'v': '/tmp/ext_v.mkv', 'streams': []}
ext_widgets = []

# ─────────────────────────────────────────────
# 7. MUXER LOGIC
# ─────────────────────────────────────────────
def analyze_swp(b):
    v = w_swp_v.value.strip()
    a = w_swp_a.value.strip()
    if not v: log('❌ Video URL required.', 'err'); return
    b.disabled = True; b.description = 'ANALYZING...'
    set_status('Downloading video...', 10)
    log('⚡ HyperSpeed Download: Video Source', 'head', clear=True)

    # Smart cache: skip if file exists and is non-empty
    if os.path.exists(state_swp['v']) and os.path.getsize(state_swp['v']) > 0:
        log(f'💾 Cache hit: {state_swp["v"]} ({file_info(state_swp["v"])})', 'warn')
    else:
        yt_download(v, state_swp['v'], get_subs=True)

    state_swp['has_ext'] = False
    if a:
        set_status('Downloading external audio...', 40)
        log('⚡ HyperSpeed Download: External Audio')
        yt_download(a, state_swp['a'])
        state_swp['has_ext'] = True

    state_swp['streams'] = []
    set_status('Probing metadata...', 70)
    log('🔎 Probing tracks (Audio + Subtitles)...')

    try:
        data_v = probe_file(state_swp['v'])
        w_swp_global.value = data_v['format'].get('tags', {}).get('title', '')
        for s in data_v.get('streams', []):
            if s['codec_type'] in ('audio', 'subtitle'):
                state_swp['streams'].append({
                    'source': 0, 'index': s['index'],
                    'type': s['codec_type'],
                    'lang': s.get('tags', {}).get('language', 'und'),
                    'name': get_track_name(s, s['index']),
                    'codec_info': get_codec_info(s)
                })
    except Exception as e:
        log(f'❌ Video probe failed: {e}', 'err')

    if state_swp['has_ext']:
        try:
            data_a = probe_file(state_swp['a'])
            for s in data_a.get('streams', []):
                if s['codec_type'] == 'audio':
                    state_swp['streams'].append({
                        'source': 1, 'index': s['index'], 'type': 'audio',
                        'lang': s.get('tags', {}).get('language', 'und'),
                        'name': f'EXT: {get_track_name(s, s["index"])}',
                        'codec_info': get_codec_info(s)
                    })
        except Exception as e:
            log(f'❌ Audio probe failed: {e}', 'err')

    build_tracks_ui(w_swp_box, swp_widgets, state_swp['streams'])
    n = len(state_swp['streams'])
    set_status(f'{n} track(s) found — edit & mux ✓', 100)
    log(f'✅ {n} tracks ready for editing.', 'ok')
    b.disabled = False; b.description = 'ANALYZE SOURCES'

def run_swapper(b, mode='full'):
    out_name = w_swp_out.value.strip() or 'MuxLab_Output'
    folder   = w_swp_folder.value.strip() or 'MuxLab_Output'
    b.disabled = True
    set_status('Muxing...', 20)
    log(f'🚀 Muxing ({mode.upper()})...', 'head', clear=True)

    cmd_in = f'-fflags +genpts -i "{state_swp["v"]}"'
    if state_swp['has_ext']:
        cmd_in += f' -i "{state_swp["a"]}"'

    selected = sorted(
        [t for t in swp_widgets if t['chk'].value],
        key=lambda x: x['pos'].value
    )

    cmd_map  = '-map 0:v:0'
    cmd_meta = f'-metadata title="{w_swp_global.value}"'
    aud_idx = sub_idx = 0

    for t in selected:
        s = t['stream']
        cmd_map += f" -map {s['source']}:{s['index']}"
        if s['type'] == 'audio':
            cmd_meta += (f' -metadata:s:a:{aud_idx} language={t["lang"].value}'
                         f' -metadata:s:a:{aud_idx} title="{t["title"].value}"')
            disp = 'default' if aud_idx == 0 else '0'
            cmd_meta += f' -disposition:a:{aud_idx} {disp}'
            aud_idx += 1
        elif s['type'] == 'subtitle':
            cmd_meta += (f' -metadata:s:s:{sub_idx} language={t["lang"].value}'
                         f' -metadata:s:s:{sub_idx} title="{t["title"].value}"')
            sub_idx += 1

    chapter_opt = '' if w_swp_chapters.value == 'Keep' else '-map_chapters -1'
    t_opt = '-t 60' if mode == 'sample' else ''
    final = f"{out_name}.mkv"

    rc = run_live(
        f'ffmpeg -y {cmd_in} {t_opt} {cmd_map} '
        f'-c:v copy -c:a copy -c:s copy '
        f'{chapter_opt} {cmd_meta} '
        f'-avoid_negative_ts make_zero "{final}"'
    )

    set_status('Uploading output...', 70)
    finish_task(final, w_swp_dest.value, folder)
    threading.Thread(target=auto_clear_swp, daemon=True).start()
    b.disabled = False

# ─────────────────────────────────────────────
# 8. EXTRACTOR LOGIC
# ─────────────────────────────────────────────
def analyze_ext(b):
    url = w_ext_url.value.strip()
    if not url: log('❌ URL required.', 'err'); return
    b.disabled = True; b.description = 'DOWNLOADING...'
    set_status('HyperSpeed download...', 10)
    log('⚡ HyperSpeed Download: Extracting Source', 'head', clear=True)

    if os.path.exists(state_ext['v']) and os.path.getsize(state_ext['v']) > 0:
        log(f'💾 Cache hit: {state_ext["v"]} ({file_info(state_ext["v"])})', 'warn')
    else:
        yt_download(url, state_ext['v'], get_subs=True)

    state_ext['streams'] = []
    set_status('Probing tracks...', 70)
    try:
        data = probe_file(state_ext['v'])
        for s in data.get('streams', []):
            if s['codec_type'] in ('audio', 'subtitle'):
                state_ext['streams'].append({
                    'source': 0, 'index': s['index'],
                    'type': s['codec_type'],
                    'lang': s.get('tags', {}).get('language', 'und'),
                    'name': get_track_name(s, s['index']),
                    'codec_info': get_codec_info(s)
                })
        build_tracks_ui(w_ext_box, ext_widgets, state_ext['streams'])
        n = len(state_ext['streams'])
        set_status(f'{n} track(s) found ✓', 100)
        log(f'✅ {n} tracks ready.', 'ok')
    except Exception as e:
        log(f'❌ Probe error: {e}', 'err')
    b.disabled = False; b.description = 'ANALYZE'

def run_extractor(b):
    out    = w_ext_out.value.strip() or 'Extracted_Audio'
    folder = w_ext_folder.value.strip() or 'MuxLab_Output'
    b.disabled = True
    set_status('Extracting tracks...', 20)
    log('🚀 Extracting selected tracks...', 'head', clear=True)

    selected = sorted(
        [t for t in ext_widgets if t['chk'].value],
        key=lambda x: x['pos'].value
    )
    if not selected:
        log('❌ No tracks selected.', 'err'); b.disabled = False; return

    cmd_map = ' '.join(f"-map 0:{t['stream']['index']}" for t in selected)
    final = f'{out}.mka'

    run_live(f'ffmpeg -y -i "{state_ext["v"]}" -vn {cmd_map} -c copy "{final}"')
    finish_task(final, w_ext_dest.value, folder)
    threading.Thread(target=auto_clear_ext, daemon=True).start()
    b.disabled = False

# ─────────────────────────────────────────────
# 9. DRIVE DOWNLOADER LOGIC
# ─────────────────────────────────────────────
def run_drive(b):
    u      = w_u2d_url.value.strip()
    n      = w_u2d_name.value.strip() or smart_filename(u)
    folder = w_u2d_folder.value.strip() or 'MuxLab_Downloads'
    if not u: log('❌ URL required.', 'err'); return
    b.disabled = True
    log('⚡ HyperSpeed → Drive Download', 'head', clear=True)
    set_status('Mounting drive...', 5)

    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    dest_dir = f"/content/drive/MyDrive/{folder}"
    os.makedirs(dest_dir, exist_ok=True)

    # Download to local SSD first (much faster), then move
    tmp = '/tmp/muxlab_dl'
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)

    set_status('Downloading to SSD...', 15)
    yt_download(u, f'{tmp}/{n}.%(ext)s', extra_flags='--no-part')

    found = glob.glob(f'{tmp}/*')
    if found:
        fname = os.path.basename(found[0])
        sz = file_info(found[0])
        set_status(f'Uploading {sz} to Drive...', 70)
        log(f'🚚 Transferring {fname} ({sz}) → Drive/{folder}...')
        shutil.move(found[0], os.path.join(dest_dir, fname))
        log(f'✅ Saved to Drive/{folder}/{fname}', 'ok')
        set_status('Drive upload complete ✓', 100)
    else:
        log('❌ Download produced no files.', 'err')

    shutil.rmtree(tmp, ignore_errors=True)
    w_u2d_url.value = ''; w_u2d_name.value = ''
    b.disabled = False

# ─────────────────────────────────────────────
# 10. UI — MOBILE-FIRST NEXT-GEN DESIGN
# ─────────────────────────────────────────────
CSS = """
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  :root {
    --bg0: #05080d;
    --bg1: #0c1018;
    --bg2: #111722;
    --bg3: #1a2335;
    --accent: #00d4ff;
    --accent2: #7c3aed;
    --ok: #22d3a5;
    --warn: #f59e0b;
    --err: #f87171;
    --text: #d1dde8;
    --muted: #4a6080;
    --border: #1a2535;
    --r: 8px;
  }

  /* Base app shell */
  .muxlab-app {
    font-family: 'Syne', sans-serif;
    background: var(--bg0);
    color: var(--text);
    max-width: 860px;
    margin: 0 auto;
    border-radius: 14px;
    border: 1px solid var(--border);
    overflow: hidden;
    box-shadow: 0 0 60px #00d4ff08, 0 0 120px #7c3aed06;
  }

  /* Masthead */
  .muxlab-head {
    background: linear-gradient(135deg, #0c1018 60%, #0f1a2e);
    border-bottom: 1px solid var(--border);
    padding: 14px 20px 12px;
    display: flex;
    align-items: center;
    gap: 10px;
    flex-wrap: wrap;
  }
  .muxlab-logo {
    font-size: 18px;
    font-weight: 800;
    letter-spacing: -.02em;
    color: #fff;
    line-height: 1;
  }
  .muxlab-logo span { color: var(--accent); }
  .muxlab-badge {
    font-family: 'JetBrains Mono', monospace;
    font-size: 9px;
    background: var(--accent2);
    color: #fff;
    padding: 2px 7px;
    border-radius: 20px;
    letter-spacing: .05em;
    font-weight: 500;
  }
  .muxlab-sub {
    font-size: 10px;
    color: var(--muted);
    font-family: 'JetBrains Mono', monospace;
    margin-left: auto;
  }

  /* Status bar */
  .muxlab-statusbar {
    background: #080b0f;
    border-bottom: 1px solid var(--border);
    padding: 4px 12px 6px;
  }

  /* Tab nav */
  .widget-tab .p-TabBar-tab {
    font-family: 'Syne', sans-serif !important;
    font-size: 10px !important;
    font-weight: 700 !important;
    letter-spacing: .08em !important;
    color: var(--muted) !important;
    background: var(--bg1) !important;
    border: none !important;
    padding: 8px 16px !important;
    text-transform: uppercase;
    transition: color .15s;
  }
  .widget-tab .p-TabBar-tab.p-mod-current {
    color: var(--accent) !important;
    background: var(--bg2) !important;
    border-bottom: 2px solid var(--accent) !important;
  }
  .widget-tab .p-TabBar-tab:hover {
    color: var(--text) !important;
  }
  .widget-tab > .p-TabBar { background: var(--bg1) !important; border-bottom: 1px solid var(--border) !important; }

  /* Inputs */
  .widget-text input,
  .widget-dropdown select,
  .widget-bounded-int-text input,
  .widget-int-text input {
    background: var(--bg3) !important;
    border: 1px solid var(--border) !important;
    color: var(--text) !important;
    border-radius: 6px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px !important;
    padding: 8px 10px !important;
    transition: border-color .15s;
  }
  .widget-text input:focus,
  .widget-dropdown select:focus {
    border-color: var(--accent) !important;
    outline: none !important;
    box-shadow: 0 0 0 2px #00d4ff18 !important;
  }

  /* Buttons */
  .widget-button.mod-primary button {
    background: linear-gradient(135deg, #0a4a6b, #0d3d5c) !important;
    border: 1px solid #00d4ff44 !important;
    color: var(--accent) !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 700 !important;
    font-size: 10px !important;
    letter-spacing: .08em !important;
    border-radius: 7px !important;
    padding: 9px 0 !important;
    transition: all .15s;
  }
  .widget-button.mod-primary button:hover {
    background: linear-gradient(135deg, #0d5c84, #0f4d70) !important;
    border-color: var(--accent) !important;
    box-shadow: 0 0 16px #00d4ff22 !important;
  }
  .widget-button.mod-success button {
    background: linear-gradient(135deg, #0a4d35, #0b3d2a) !important;
    border: 1px solid #22d3a544 !important;
    color: var(--ok) !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 700 !important;
    font-size: 10px !important;
    letter-spacing: .1em !important;
    border-radius: 7px !important;
    padding: 9px 0 !important;
    transition: all .15s;
  }
  .widget-button.mod-success button:hover {
    background: linear-gradient(135deg, #0f6b48, #0c4f37) !important;
    border-color: var(--ok) !important;
    box-shadow: 0 0 16px #22d3a522 !important;
  }
  .widget-button.mod-warning button {
    background: linear-gradient(135deg, #4d3a0a, #3d2d0b) !important;
    border: 1px solid #f59e0b44 !important;
    color: var(--warn) !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 700 !important;
    font-size: 10px !important;
    letter-spacing: .08em !important;
    border-radius: 7px !important;
    padding: 9px 0 !important;
  }

  /* Toggle buttons */
  .widget-toggle-buttons button {
    font-family: 'Syne', sans-serif !important;
    font-size: 10px !important;
    font-weight: 600 !important;
    background: var(--bg3) !important;
    color: var(--muted) !important;
    border: 1px solid var(--border) !important;
  }
  .widget-toggle-buttons button.mod-active {
    background: var(--bg2) !important;
    color: var(--accent) !important;
    border-color: var(--accent) !important;
  }

  /* Checkbox */
  .widget-checkbox input[type=checkbox] { accent-color: var(--accent); }

  /* Tab content padding */
  .widget-tab > .widget-tab-contents { background: var(--bg1) !important; padding: 14px !important; }

  /* Section label */
  .sec-label {
    font-size: 9px;
    font-weight: 700;
    letter-spacing: .12em;
    text-transform: uppercase;
    color: var(--muted);
    padding: 10px 0 4px;
    border-top: 1px solid var(--border);
    margin-top: 6px;
  }
  .sec-label:first-child { border-top: none; padding-top: 0; margin-top: 0; }

  /* Divider */
  .mux-divider {
    height: 1px;
    background: var(--border);
    margin: 10px 0;
  }

  /* Log console */
  .widget-output {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 11px !important;
    background: #05080d !important;
    border-radius: 0 0 12px 12px !important;
  }

  /* Progress */
  .widget-progress .progress-bar { transition: width .3s ease !important; }

  /* Mobile tweaks */
  @media (max-width: 600px) {
    .muxlab-sub { display: none; }
    .widget-tab .p-TabBar-tab { padding: 8px 10px !important; font-size: 9px !important; }
    .widget-tab > .widget-tab-contents { padding: 10px !important; }
  }
</style>
"""

full   = widgets.Layout(width='100%')
half   = widgets.Layout(width='50%')
pad    = widgets.Layout(margin='4px 0')

def section(txt):
    return widgets.HTML(f"<div class='sec-label'>{txt}</div>")

def divider():
    return widgets.HTML("<div class='mux-divider'></div>")

# ── TAB 1: MUXER ─────────────────────────────
w_swp_v       = widgets.Text(placeholder='🎬  Video / Stream URL', layout=full)
w_swp_a       = widgets.Text(placeholder='🎵  External Audio URL  (optional)', layout=full)
w_swp_an      = widgets.Button(description='⚡  ANALYZE SOURCES', button_style='primary', layout=full)
w_swp_an.on_click(analyze_swp)
w_swp_global  = widgets.Text(placeholder='Global Title (optional)', layout=full)
w_swp_box     = widgets.VBox([], layout=widgets.Layout(min_height='40px'))
w_swp_out     = widgets.Text(placeholder='Output Filename  (no ext)', layout=full)
w_swp_folder  = widgets.Text(placeholder='Drive Folder  (default: MuxLab_Output)', layout=full)
w_swp_dest    = widgets.ToggleButtons(
    options=['Local', 'Drive'], value='Local',
    layout=widgets.Layout(width='auto')
)
w_swp_chapters = widgets.ToggleButtons(
    options=['Keep', 'Strip'], value='Keep',
    layout=widgets.Layout(width='auto'),
    description='Chapters:'
)
w_swp_run     = widgets.Button(description='🚀  MUX FULL', button_style='success', layout=full)
w_swp_sample  = widgets.Button(description='🔬  MUX SAMPLE  (60s)', button_style='warning',
                                layout=widgets.Layout(width='100%'))
w_swp_run.on_click(lambda b: run_swapper(b, 'full'))
w_swp_sample.on_click(lambda b: run_swapper(b, 'sample'))

tab1 = widgets.VBox([
    section('SOURCE URLS'),
    w_swp_v, w_swp_a, w_swp_an,
    section('GLOBAL METADATA'),
    w_swp_global,
    section('TRACK EDITOR'),
    w_swp_box,
    section('OUTPUT OPTIONS'),
    w_swp_out,
    widgets.HBox([
        widgets.VBox([widgets.HTML("<span style='font-size:9px;color:#4a6080;font-weight:700;letter-spacing:.08em;text-transform:uppercase'>Destination</span>"), w_swp_dest]),
        widgets.VBox([widgets.HTML("<span style='font-size:9px;color:#4a6080;font-weight:700;letter-spacing:.08em;text-transform:uppercase'>Chapters</span>"), w_swp_chapters]),
    ], layout=widgets.Layout(gap='20px', align_items='flex-start', flex_wrap='wrap')),
    w_swp_folder,
    divider(),
    w_swp_run,
    widgets.HTML("<div style='height:4px'></div>"),
    w_swp_sample,
], layout=widgets.Layout(gap='3px'))

# ── TAB 2: EXTRACTOR ─────────────────────────
w_ext_url    = widgets.Text(placeholder='🎬  Video / Stream URL', layout=full)
w_ext_an     = widgets.Button(description='⚡  ANALYZE', button_style='primary', layout=full)
w_ext_an.on_click(analyze_ext)
w_ext_box    = widgets.VBox([], layout=widgets.Layout(min_height='40px'))
w_ext_out    = widgets.Text(placeholder='Output Filename  (no ext)', layout=full)
w_ext_folder = widgets.Text(placeholder='Drive Folder  (default: MuxLab_Output)', layout=full)
w_ext_dest   = widgets.ToggleButtons(
    options=['Local', 'Drive'], value='Local',
    layout=widgets.Layout(width='auto')
)
w_ext_run    = widgets.Button(description='🚀  EXTRACT TRACKS', button_style='success', layout=full)
w_ext_run.on_click(run_extractor)

tab2 = widgets.VBox([
    section('SOURCE URL'),
    w_ext_url, w_ext_an,
    section('TRACK SELECTOR'),
    w_ext_box,
    section('OUTPUT OPTIONS'),
    w_ext_out,
    widgets.HBox([
        widgets.VBox([widgets.HTML("<span style='font-size:9px;color:#4a6080;font-weight:700;letter-spacing:.08em;text-transform:uppercase'>Destination</span>"), w_ext_dest]),
    ]),
    w_ext_folder,
    divider(),
    w_ext_run,
], layout=widgets.Layout(gap='3px'))

# ── TAB 3: DRIVE DOWNLOADER ──────────────────
w_u2d_url    = widgets.Text(placeholder='🔗  Direct / YouTube / Any yt-dlp URL', layout=full)
w_u2d_name   = widgets.Text(placeholder='Custom Filename  (no ext, optional)', layout=full)
w_u2d_folder = widgets.Text(placeholder='Drive Folder  (default: MuxLab_Downloads)', layout=full)
w_u2d_run    = widgets.Button(description='⚡  HYPERSPEED DOWNLOAD → DRIVE',
                               button_style='success', layout=full)
w_u2d_run.on_click(run_drive)

tab3 = widgets.VBox([
    section('SOURCE'),
    w_u2d_url, w_u2d_name,
    section('DESTINATION'),
    w_u2d_folder,
    divider(),
    w_u2d_run,
], layout=widgets.Layout(gap='3px'))

# ── TABS ─────────────────────────────────────
tabs = widgets.Tab(children=[tab1, tab2, tab3])
for i, t in enumerate(['MUXER', 'EXTRACTOR', 'DRIVE DL']):
    tabs.set_title(i, t)

# ── STATUS BAR ───────────────────────────────
statusbar = widgets.VBox([
    progress_bar, status_lbl
], layout=widgets.Layout(
    padding='4px 12px 6px',
    background_color='#080b0f',
    border_bottom='1px solid #1a2535'
))

# ── MASTHEAD ─────────────────────────────────
masthead = widgets.HTML("""
<div class="muxlab-head">
  <span class="muxlab-logo">Mux<span>Lab</span></span>
  <span class="muxlab-badge">V4 TITANIUM ULTRA</span>
  <span class="muxlab-sub">⚡ 32-conn HyperSpeed &nbsp;|&nbsp; Mobile-First &nbsp;|&nbsp; Smart Cache &nbsp;|&nbsp; Parallel Upload</span>
</div>
""")

# ── FULL APP ─────────────────────────────────
app = widgets.VBox([
    widgets.HTML(CSS),
    masthead,
    statusbar,
    tabs,
    log_box,
])
app.add_class('muxlab-app')

# ── BOOT ─────────────────────────────────────
log('🔧 Installing dependencies...', 'head')
install_deps('Stable')
clear_output(wait=True)
display(app)
